<a href="https://colab.research.google.com/github/marshallpitts/movie-recommender-svd/blob/main/notebooks/02_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import os, urllib.request, zipfile
import numpy as np
import pandas as pd

if not os.path.exists("ml-100k"):
    urllib.request.urlretrieve(
        "https://files.grouplens.org/datasets/movielens/ml-100k.zip",
        "ml-100k.zip")
    zipfile.ZipFile("ml-100k.zip").extractall()

ratings = pd.read_csv(
    "ml-100k/u.data",
    sep="\t",
    names=["user", "item", "rating", "time"])

print(ratings.head())
print("Number of ratings:", len(ratings))


   user  item  rating       time
0   196   242       3  881250949
1   186   302       3  891717742
2    22   377       1  878887116
3   244    51       2  880606923
4   166   346       1  886397596
Number of ratings: 100000


This code downloads the dataset from MovieLens and loads it into a Pandas DataFrame. It contains 100,000 movie ratings, including the user ID, movie ID, rating and timestamp. It also checks whether the dataset already exists to avoid downloading it multiple times.

In [2]:
def split_ratings(ratings, test_frac=0.10, seed=0):
    """Hide ratings without removing a user's or movie's last training rating."""
    rng = np.random.default_rng(seed)
    is_test = np.zeros(len(ratings), dtype=bool)

    user_left = ratings.groupby("user").size().to_dict()
    item_left = ratings.groupby("item").size().to_dict()

    for pos in rng.permutation(len(ratings)):
        row = ratings.iloc[pos]
        user, item = row["user"], row["item"]

        if (rng.random() < test_frac
                and user_left[user] > 1
                and item_left[item] > 1):
            is_test[pos] = True
            user_left[user] -= 1
            item_left[item] -= 1

    return ratings.iloc[~is_test].copy(), ratings.iloc[is_test].copy()

train, test = split_ratings(ratings)

print("train:", len(train), "test:", len(test))
print("test fraction:", round(len(test) / len(ratings), 3))

train: 90032 test: 9968
test fraction: 0.1


The code randomly divides the movie ratings into training and testing sets. Around 90% of the ratings are used for training while the rest are hidden for testing. The code also makes sure that every user and movie has at least one rating in the set being used.

In [3]:
N_USERS, N_ITEMS = 943, 1682

def to_matrix(df):
    """Return a user-by-movie matrix; missing ratings are np.nan."""
    R = np.full((N_USERS, N_ITEMS), np.nan)

    R[df["user"].to_numpy() - 1,
      df["item"].to_numpy() - 1] = df["rating"].to_numpy()

    return R

R = to_matrix(train)
known = ~np.isnan(R)

print("Matrix shape:", R.shape)
print("Known training ratings:", known.sum())

Matrix shape: (943, 1682)
Known training ratings: 90032


The code converts the training ratings into a matrix where each row represents a user and each column represents a movie.

In [4]:
truth = test["rating"].to_numpy()
u = test["user"].to_numpy() - 1
i = test["item"].to_numpy() - 1

This code prepares the hidden test ratings for evaluation. The truth array includes the actual ratings and u and i store the corresponding user and movie positions. These values are used to compare our predictions to the ratings.

In [5]:
def rmse(pred, truth):
    """Root mean squared error, measured in stars."""
    return np.sqrt(np.mean((pred - truth)**2))

This function calculates the root mean squared error between predicted and actual movie ratings.

In [6]:
mu = np.nanmean(R)
user_mean = np.nanmean(R, axis=1)

n_item = known.sum(axis=0)
item_mean = np.full(N_ITEMS, mu)
has_rating = n_item > 0

item_mean[has_rating] = (
    np.nansum(R, axis=0)[has_rating] / n_item[has_rating]
)

print("Overall average:", round(mu, 3))

Overall average: 3.53


This code calculates the overall average rating given by each user and average rating received by each movie. Missing ratings are ignored and the averages are used to predict ratings in the test set.

In [7]:
scores = {}

scores["always 3"] = rmse(
    np.full(len(test), 3.0), truth
)

scores["overall average"] = rmse(
    np.full(len(test), mu), truth
)

scores["user average"] = rmse(
    user_mean[u], truth
)

scores["movie average"] = rmse(
    item_mean[i], truth
)

for name, score in scores.items():
    print(f"{name:16s} {score:.4f}")

always 3         1.2516
overall average  1.1349
user average     1.0426
movie average    1.0284


This block evaluates 4 differnt baseline recommendation methods using the same hidden test ratings. Each method predicts ratings differently and the RSME function measures its accuracy. Comparing these scores helps determine which method performs best.

In [8]:
def bias_baseline(R):
    """Overall mean plus one user effect and one movie effect."""
    known = ~np.isnan(R)
    mu = np.nanmean(R)

    b_user = np.nanmean(R - mu, axis=1)
    residual = R - mu - b_user[:, None]

    n_item = known.sum(axis=0)
    b_item = np.zeros(R.shape[1])
    has_rating = n_item > 0

    b_item[has_rating] = (
        np.nansum(residual, axis=0)[has_rating] / n_item[has_rating]
    )

    return mu, b_user, b_item

The function creates a prediction model that considers both user and movie rating patterns. It calculates the overall average, then determines how much each user's ratings differ from that average. Finally, it calculates each movie's remaining effect after taking in consideration user differences.

In [9]:
mu, b_user, b_item = bias_baseline(R)

pred = np.clip(
    mu + b_user[u] + b_item[i], 1, 5
)

scores["user + movie effects"] = rmse(pred, truth)

print(
    pd.Series(scores, name="test RMSE")
    .round(4)
    .sort_values()
)

user + movie effects    0.9545
movie average           1.0284
user average            1.0426
overall average         1.1349
always 3                1.2516
Name: test RMSE, dtype: float64


This code uses the combined user and movie effects to predict the hidden ratings. The predictions are limited to 1-5 stars. The RMSE is calculated and compared with the four earlier models to decide which method makes the most accurate predictions.

Excercise 1

In [10]:
tiny_R = np.array([
    [5, 3, np.nan],
    [4, np.nan, 1],
    [np.nan, 2, 2]
])

tiny_mu, tiny_b_user, tiny_b_item = bias_baseline(tiny_R)

print("Overall mean:", tiny_mu)
print("User effects:", tiny_b_user)
print("Movie effects:", tiny_b_item)

Overall mean: 2.8333333333333335
User effects: [ 1.16666667 -0.33333333 -0.83333333]
Movie effects: [ 1.25 -0.5  -0.75]


The overall mean of the known ratings is 2.8333. This was calculated by subtracting the overall mean from each user's average rating. Then calculating the movie effects using the remaining differences.

Excercise 2

In [11]:
R_all = to_matrix(ratings)

cheating_user_mean = np.nanmean(R_all, axis=1)

cheating_rmse = rmse(cheating_user_mean[u], truth)

print("Honest user-average RMSE:", scores["user average"])
print("Cheating user-average RMSE:", cheating_rmse)

Honest user-average RMSE: 1.0425731518955943
Cheating user-average RMSE: 1.0322056951414382


The honest user-average model had an RSME of 1.0426, while the cheating model had 1.0322. This is due to the cheating model using all the ratings (including the test ratings that were hidden). Since the model already had access to the data it was being tested on, its predictions seemed more accurate. This is an example of data leakage, and the lower RSME is misleading because it doesnt accurately reflect how well the model would predict ratings it's never seen.

Excercise 3

In [12]:
train1, test1 = split_ratings(ratings, seed=1)

R1 = to_matrix(train1)
known1 = ~np.isnan(R1)

truth1 = test1["rating"].to_numpy()
u1 = test1["user"].to_numpy() - 1
i1 = test1["item"].to_numpy() - 1

mu1 = np.nanmean(R1)
user_mean1 = np.nanmean(R1, axis=1)

n_item1 = known1.sum(axis=0)
item_mean1 = np.full(N_ITEMS, mu1)
has_rating1 = n_item1 > 0

item_mean1[has_rating1] = (
    np.nansum(R1, axis=0)[has_rating1] / n_item1[has_rating1]
)

mu_bias1, b_user1, b_item1 = bias_baseline(R1)

pred1 = np.clip(
    mu_bias1 + b_user1[u1] + b_item1[i1], 1, 5
)

scores1 = {
    "always 3": rmse(np.full(len(test1), 3.0), truth1),
    "overall average": rmse(np.full(len(test1), mu1), truth1),
    "user average": rmse(user_mean1[u1], truth1),
    "movie average": rmse(item_mean1[i1], truth1),
    "user + movie effects": rmse(pred1, truth1)
}

comparison = pd.DataFrame({
    "Seed 0": pd.Series(scores),
    "Seed 1": pd.Series(scores1)
})

print(comparison.round(4))

                      Seed 0  Seed 1
always 3              1.2516  1.2336
overall average       1.1349  1.1127
user average          1.0426  1.0345
movie average         1.0284  1.0159
user + movie effects  0.9545  0.9577


The RSME values for all models changed slightly after changing the random seed from 0 to 1. Most models had a lower RMSE with seed 1, except for the user and movie effects model. However, the ranking of the models stayed the same. The user and movie effects model performed the best with both seeds, while the always 3 model had the highest RMSE.

These results show that model performance can vary depending on how data is randomly split into training and testing sets. Even though the scores changed, the overall ranking remained consistent. This shows why it is important to compare recommendation models using the same training and testing data to ensure a fair comparison.